# 🔬 Laboratory Testing Services Platform
## Backend Database Setup & CRUD Operations

**Project:** Integrated Search and Comparison Platform for Specialized Laboratory Testing Services in Indonesia

**Database:** 9 Tables with MySQL

---

## CELL 1: Install Dependencies

In [9]:
import subprocess
import sys

packages = ['mysql-connector-python', 'pandas', 'python-dotenv']

for package in packages:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
        print(f'✅ {package} installed')
    except:
        print(f'⚠️ {package} may already be installed')

print('\n✅ All dependencies ready!')

✅ mysql-connector-python installed
✅ pandas installed
✅ python-dotenv installed

✅ All dependencies ready!


## CELL 2: Import Libraries

In [10]:
import mysql.connector
from mysql.connector import Error
import pandas as pd
from datetime import datetime
import json
import os
from dotenv import load_dotenv

load_dotenv()
print('✅ All libraries imported successfully')

✅ All libraries imported successfully


## CELL 3: Database Configuration

In [12]:
# Database Configuration
DB_CONFIG = {
    'host': os.getenv('DB_HOST', 'localhost'),
    'user': os.getenv('DB_USER', 'root'),
    'password': os.getenv('DB_PASSWORD', ''),
    'database': os.getenv('DB_NAME', 'laboratory_db')
}

print('📋 Database Configuration:')
print(f"  Host: {DB_CONFIG['host']}")
print(f"  User: {DB_CONFIG['user']}")
print(f"  password: {DB_CONFIG['password']}")
print(f"  Database: {DB_CONFIG['database']}")
print(f"  Password: {'***' if DB_CONFIG['password'] else '(empty)'}")

📋 Database Configuration:
  Host: localhost
  User: root
  password: root
  Database: laboratory_db
  Password: ***


## CELL 4: Database Connection Class

In [13]:
class DatabaseConnection:
    """Manage MySQL database connections and queries"""
    
    def __init__(self, config):
        self.config = config
        self.connection = None
    
    def connect(self):
        """Establish database connection"""
        try:
            self.connection = mysql.connector.connect(**self.config)
            print('✅ Connected to MySQL database')
            return True
        except Error as e:
            print(f'❌ Error: {e}')
            return False
    
    def disconnect(self):
        """Close database connection"""
        if self.connection and self.connection.is_connected():
            self.connection.close()
            print('✅ Disconnected from database')
    
    def execute_query(self, query, params=None, show_result=False):
        """Execute a query (INSERT, UPDATE, DELETE)"""
        try:
            cursor = self.connection.cursor()
            if params:
                cursor.execute(query, params)
            else:
                cursor.execute(query)
            self.connection.commit()
            affected_rows = cursor.rowcount
            if show_result:
                print(f'✅ Query executed ({affected_rows} rows affected)')
            cursor.close()
            return True
        except Error as e:
            print(f'❌ Error: {e}')
            return False
    
    def fetch_query(self, query, params=None):
        """Fetch data from database"""
        try:
            cursor = self.connection.cursor(dictionary=True)
            if params:
                cursor.execute(query, params)
            else:
                cursor.execute(query)
            result = cursor.fetchall()
            cursor.close()
            return result
        except Error as e:
            print(f'❌ Error: {e}')
            return None
    
    def execute_multiple(self, queries):
        """Execute multiple queries"""
        try:
            cursor = self.connection.cursor()
            for query in queries:
                if query.strip():
                    cursor.execute(query)
            self.connection.commit()
            print(f'✅ Executed {len([q for q in queries if q.strip()])} queries')
            cursor.close()
            return True
        except Error as e:
            print(f'❌ Error: {e}')
            return False

# Initialize database connection
db = DatabaseConnection(DB_CONFIG)
print('✅ Database connection class created')

✅ Database connection class created


## CELL 5: Connect to MySQL

In [14]:
# Test 1: Cek apakah .env file exists
import os
env_path = r"C:\Users\fadhi\Documents\Laboratory_Platform\.env"
print(f"File .env ada? {os.path.exists(env_path)}")
print(f"Path: {env_path}")

File .env ada? True
Path: C:\Users\fadhi\Documents\Laboratory_Platform\.env


In [15]:
import mysql.connector

connection = mysql.connector.connect(
    host='localhost',
    user='root',
    password='root',
    database='mysql'
)
print("✅ Connect berhasil!")
connection.close()

✅ Connect berhasil!


In [19]:
print(f"db object: {db}")
print(f"db.connection: {db.connection}")

db object: <__main__.DatabaseConnection object at 0x0000028EEE5E26D0>
db.connection: None


In [20]:
# Connect to MySQL server
if db.connect():
    print('✅ Connected to MySQL database')
else:
    print('❌ Failed to connect. Check your MySQL server and .env file')
    print(f'Config: {DB_CONFIG}')

❌ Error: 1049 (42000): Unknown database 'laboratory_db'
❌ Failed to connect. Check your MySQL server and .env file
Config: {'host': 'localhost', 'user': 'root', 'password': 'root', 'database': 'laboratory_db'}


In [17]:
# Connect to MySQL server
if db.connect():
    print('✅ Ready to setup database')
else:
    print('❌ Failed to connect. Check your MySQL server and .env file')

❌ Error: 1049 (42000): Unknown database 'laboratory_db'
❌ Failed to connect. Check your MySQL server and .env file


## CELL 6: Create Database

In [21]:
import mysql.connector

# Connect ke database DEFAULT (mysql), bukan laboratory_db
conn = mysql.connector.connect(
    host='localhost',
    user='root',
    password='root'
)

db.connection = conn
print('✅ Connected to MySQL (default database)')

✅ Connected to MySQL (default database)


In [22]:
# Create database if not exists
create_db_query = "CREATE DATABASE IF NOT EXISTS laboratory_db"
cursor = db.connection.cursor()
cursor.execute(create_db_query)
db.connection.commit()
cursor.close()
print('✅ Database created/verified: laboratory_db')

✅ Database created/verified: laboratory_db


In [26]:
import mysql.connector

# Sekarang connect ke laboratory_db yang sudah dibuat
conn = mysql.connector.connect(
    host='localhost',
    user='root',
    password='root',
    database='laboratory_db'
)

db.connection = conn
print('✅ Connected to laboratory_db')

✅ Connected to laboratory_db


## CELL 7: Create All 9 Tables

In [27]:
# SQL to create all 9 tables
create_tables_sql = [
    # 1. User Table
    """
    CREATE TABLE IF NOT EXISTS user (
        user_id INT PRIMARY KEY AUTO_INCREMENT,
        username VARCHAR(100) UNIQUE NOT NULL,
        email VARCHAR(100) UNIQUE NOT NULL,
        password_hash VARCHAR(255) NOT NULL,
        full_name VARCHAR(150),
        organization VARCHAR(150),
        phone VARCHAR(20),
        user_role ENUM('researcher', 'industry', 'admin') DEFAULT 'researcher',
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP
    )
    """,
    
    # 2. Laboratory Table
    """
    CREATE TABLE IF NOT EXISTS laboratory (
        lab_id INT PRIMARY KEY AUTO_INCREMENT,
        lab_name VARCHAR(200) NOT NULL,
        description TEXT,
        contact_email VARCHAR(100),
        contact_phone VARCHAR(20),
        website VARCHAR(255),
        established_year INT,
        operational_status ENUM('operational', 'maintenance', 'closed') DEFAULT 'operational',
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP
    )
    """,
    
    # 3. Location Sorter Table
    """
    CREATE TABLE IF NOT EXISTS location_sorter (
        location_id INT PRIMARY KEY AUTO_INCREMENT,
        lab_id INT NOT NULL,
        province VARCHAR(100) NOT NULL,
        city VARCHAR(100) NOT NULL,
        district VARCHAR(100),
        address TEXT,
        postal_code VARCHAR(10),
        latitude DECIMAL(10, 8),
        longitude DECIMAL(11, 8),
        FOREIGN KEY (lab_id) REFERENCES laboratory(lab_id) ON DELETE CASCADE,
        INDEX idx_province (province),
        INDEX idx_city (city)
    )
    """,
    
    # 4. Service Table
    """
    CREATE TABLE IF NOT EXISTS service (
        service_id INT PRIMARY KEY AUTO_INCREMENT,
        service_name VARCHAR(200) NOT NULL,
        description TEXT,
        service_category VARCHAR(100),
        estimated_turnaround_days INT,
        base_price DECIMAL(12, 2),
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        INDEX idx_category (service_category)
    )
    """,
    
    # 5. Lab-Service Junction Table
    """
    CREATE TABLE IF NOT EXISTS lab_service_junction (
        junction_id INT PRIMARY KEY AUTO_INCREMENT,
        lab_id INT NOT NULL,
        service_id INT NOT NULL,
        available BOOLEAN DEFAULT TRUE,
        service_price DECIMAL(12, 2),
        current_capacity INT DEFAULT 0,
        max_capacity INT,
        FOREIGN KEY (lab_id) REFERENCES laboratory(lab_id) ON DELETE CASCADE,
        FOREIGN KEY (service_id) REFERENCES service(service_id) ON DELETE CASCADE,
        UNIQUE KEY unique_lab_service (lab_id, service_id)
    )
    """,
    
    # 6. Equipment Table
    """
    CREATE TABLE IF NOT EXISTS equipment (
        equipment_id INT PRIMARY KEY AUTO_INCREMENT,
        lab_id INT NOT NULL,
        equipment_name VARCHAR(200) NOT NULL,
        equipment_type VARCHAR(100),
        manufacturer VARCHAR(150),
        model VARCHAR(100),
        acquisition_date DATE,
        operational_status ENUM('operational', 'maintenance', 'inactive') DEFAULT 'operational',
        FOREIGN KEY (lab_id) REFERENCES laboratory(lab_id) ON DELETE CASCADE,
        INDEX idx_lab (lab_id)
    )
    """,
    
    # 7. Certification Table
    """
    CREATE TABLE IF NOT EXISTS certification (
        certification_id INT PRIMARY KEY AUTO_INCREMENT,
        lab_id INT NOT NULL,
        certification_name VARCHAR(150) NOT NULL,
        issuing_body VARCHAR(150),
        issue_date DATE,
        expiry_date DATE,
        certification_status ENUM('valid', 'expired', 'suspended') DEFAULT 'valid',
        FOREIGN KEY (lab_id) REFERENCES laboratory(lab_id) ON DELETE CASCADE,
        INDEX idx_lab (lab_id)
    )
    """,
    
    # 8. Accreditation Table
    """
    CREATE TABLE IF NOT EXISTS accreditation (
        accreditation_id INT PRIMARY KEY AUTO_INCREMENT,
        lab_id INT NOT NULL,
        accreditation_name VARCHAR(150) NOT NULL,
        accrediting_body VARCHAR(150),
        standard_code VARCHAR(50),
        accreditation_scope TEXT,
        issue_date DATE,
        expiry_date DATE,
        accreditation_status ENUM('active', 'inactive', 'suspended') DEFAULT 'active',
        FOREIGN KEY (lab_id) REFERENCES laboratory(lab_id) ON DELETE CASCADE,
        INDEX idx_lab (lab_id)
    )
    """,
    
    # 9. Sample Table
    """
    CREATE TABLE IF NOT EXISTS sample (
        sample_id INT PRIMARY KEY AUTO_INCREMENT,
        user_id INT NOT NULL,
        lab_id INT NOT NULL,
        service_id INT NOT NULL,
        sample_description TEXT,
        sample_type VARCHAR(100),
        submission_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        expected_completion_date DATE,
        sample_status ENUM('submitted', 'processing', 'completed', 'failed') DEFAULT 'submitted',
        result_notes TEXT,
        cost DECIMAL(12, 2),
        FOREIGN KEY (user_id) REFERENCES user(user_id) ON DELETE CASCADE,
        FOREIGN KEY (lab_id) REFERENCES laboratory(lab_id) ON DELETE CASCADE,
        FOREIGN KEY (service_id) REFERENCES service(service_id) ON DELETE CASCADE,
        INDEX idx_user (user_id),
        INDEX idx_lab (lab_id)
    )
    """
]

# Execute all table creation queries
cursor = db.connection.cursor()
for i, query in enumerate(create_tables_sql, 1):
    try:
        cursor.execute(query)
        db.connection.commit()
        print(f'✅ Table {i}/9 created')
    except Error as e:
        print(f'⚠️ Table {i}: {str(e)[:60]}')
cursor.close()
print('\n✅ All 9 tables ready!')

✅ Table 1/9 created
✅ Table 2/9 created
✅ Table 3/9 created
✅ Table 4/9 created
✅ Table 5/9 created
✅ Table 6/9 created
✅ Table 7/9 created
✅ Table 8/9 created
✅ Table 9/9 created

✅ All 9 tables ready!


## CELL 8: CRUD Operations Class

In [28]:
class LaboratoryManager:
    """Manage laboratory testing services operations"""
    
    def __init__(self, db_connection):
        self.db = db_connection
    
    # ===== LABORATORY OPERATIONS =====
    
    def add_laboratory(self, lab_name, description, contact_email, contact_phone, website, established_year):
        """Add new laboratory"""
        query = """
        INSERT INTO laboratory (lab_name, description, contact_email, contact_phone, website, established_year)
        VALUES (%s, %s, %s, %s, %s, %s)
        """
        return self.db.execute_query(query, (lab_name, description, contact_email, contact_phone, website, established_year), show_result=True)
    
    def get_all_laboratories(self):
        """Get all laboratories"""
        query = "SELECT * FROM laboratory ORDER BY lab_name"
        result = self.db.fetch_query(query)
        return pd.DataFrame(result) if result else pd.DataFrame()
    
    def get_laboratory_by_id(self, lab_id):
        """Get laboratory details by ID"""
        query = "SELECT * FROM laboratory WHERE lab_id = %s"
        result = self.db.fetch_query(query, (lab_id,))
        return result[0] if result else None
    
    def update_laboratory(self, lab_id, **kwargs):
        """Update laboratory information"""
        allowed_fields = ['lab_name', 'description', 'contact_email', 'contact_phone', 'website', 'established_year', 'operational_status']
        updates = {k: v for k, v in kwargs.items() if k in allowed_fields and v is not None}
        if not updates:
            print('No updates provided')
            return False
        
        set_clause = ', '.join([f"{k} = %s" for k in updates.keys()])
        values = list(updates.values()) + [lab_id]
        query = f"UPDATE laboratory SET {set_clause} WHERE lab_id = %s"
        return self.db.execute_query(query, values, show_result=True)
    
    def delete_laboratory(self, lab_id):
        """Delete laboratory and related data"""
        query = "DELETE FROM laboratory WHERE lab_id = %s"
        return self.db.execute_query(query, (lab_id,), show_result=True)
    
    # ===== LOCATION OPERATIONS =====
    
    def add_location(self, lab_id, province, city, district, address, postal_code, latitude=None, longitude=None):
        """Add location for laboratory"""
        query = """
        INSERT INTO location_sorter (lab_id, province, city, district, address, postal_code, latitude, longitude)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        """
        return self.db.execute_query(query, (lab_id, province, city, district, address, postal_code, latitude, longitude), show_result=True)
    
    def get_location_by_lab(self, lab_id):
        """Get location by laboratory ID"""
        query = "SELECT * FROM location_sorter WHERE lab_id = %s"
        result = self.db.fetch_query(query, (lab_id,))
        return result[0] if result else None
    
    def search_labs_by_location(self, province=None, city=None, district=None):
        """Search laboratories by location"""
        query = """
        SELECT l.*, loc.province, loc.city, loc.district, loc.address 
        FROM laboratory l 
        LEFT JOIN location_sorter loc ON l.lab_id = loc.lab_id 
        WHERE 1=1
        """
        params = []
        
        if province:
            query += " AND loc.province = %s"
            params.append(province)
        if city:
            query += " AND loc.city = %s"
            params.append(city)
        if district:
            query += " AND loc.district = %s"
            params.append(district)
        
        query += " ORDER BY l.lab_name"
        result = self.db.fetch_query(query, params) if params else self.db.fetch_query(query)
        return pd.DataFrame(result) if result else pd.DataFrame()
    
    def get_all_provinces(self):
        """Get all unique provinces"""
        query = "SELECT DISTINCT province FROM location_sorter ORDER BY province"
        result = self.db.fetch_query(query)
        return [row['province'] for row in result] if result else []
    
    def get_cities_by_province(self, province):
        """Get all unique cities in a province"""
        query = "SELECT DISTINCT city FROM location_sorter WHERE province = %s ORDER BY city"
        result = self.db.fetch_query(query, (province,))
        return [row['city'] for row in result] if result else []
    
    # ===== SERVICE OPERATIONS =====
    
    def add_service(self, service_name, description, service_category, estimated_turnaround_days, base_price):
        """Add new service type"""
        query = """
        INSERT INTO service (service_name, description, service_category, estimated_turnaround_days, base_price)
        VALUES (%s, %s, %s, %s, %s)
        """
        return self.db.execute_query(query, (service_name, description, service_category, estimated_turnaround_days, base_price), show_result=True)
    
    def get_all_services(self):
        """Get all available services"""
        query = "SELECT * FROM service ORDER BY service_category, service_name"
        result = self.db.fetch_query(query)
        return pd.DataFrame(result) if result else pd.DataFrame()
    
    def get_service_categories(self):
        """Get all unique service categories"""
        query = "SELECT DISTINCT service_category FROM service ORDER BY service_category"
        result = self.db.fetch_query(query)
        return [row['service_category'] for row in result] if result else []
    
    # ===== LAB-SERVICE JUNCTION OPERATIONS =====
    
    def link_service_to_lab(self, lab_id, service_id, service_price, max_capacity):
        """Link service to laboratory"""
        query = """
        INSERT INTO lab_service_junction (lab_id, service_id, service_price, max_capacity)
        VALUES (%s, %s, %s, %s)
        """
        return self.db.execute_query(query, (lab_id, service_id, service_price, max_capacity), show_result=True)
    
    def get_services_by_lab(self, lab_id):
        """Get all services offered by a laboratory"""
        query = """
        SELECT s.service_id, s.service_name, s.service_category, s.estimated_turnaround_days,
               lsj.service_price, lsj.current_capacity, lsj.max_capacity, lsj.available
        FROM service s 
        JOIN lab_service_junction lsj ON s.service_id = lsj.service_id 
        WHERE lsj.lab_id = %s
        """
        result = self.db.fetch_query(query, (lab_id,))
        return pd.DataFrame(result) if result else pd.DataFrame()
    
    def search_labs_by_service(self, service_category=None, service_id=None):
        """Search laboratories by service offered"""
        query = """
        SELECT DISTINCT l.lab_id, l.lab_name, l.description, l.contact_email, l.contact_phone, 
               s.service_name, s.service_category, lsj.service_price
        FROM laboratory l
        JOIN lab_service_junction lsj ON l.lab_id = lsj.lab_id
        JOIN service s ON lsj.service_id = s.service_id
        WHERE lsj.available = TRUE
        """
        params = []
        
        if service_category:
            query += " AND s.service_category = %s"
            params.append(service_category)
        if service_id:
            query += " AND s.service_id = %s"
            params.append(service_id)
        
        query += " ORDER BY l.lab_name"
        result = self.db.fetch_query(query, params) if params else self.db.fetch_query(query)
        return pd.DataFrame(result) if result else pd.DataFrame()
    
    # ===== EQUIPMENT OPERATIONS =====
    
    def add_equipment(self, lab_id, equipment_name, equipment_type, manufacturer, model, acquisition_date):
        """Add equipment to laboratory"""
        query = """
        INSERT INTO equipment (lab_id, equipment_name, equipment_type, manufacturer, model, acquisition_date)
        VALUES (%s, %s, %s, %s, %s, %s)
        """
        return self.db.execute_query(query, (lab_id, equipment_name, equipment_type, manufacturer, model, acquisition_date), show_result=True)
    
    def get_equipment_by_lab(self, lab_id):
        """Get all equipment in a laboratory"""
        query = "SELECT * FROM equipment WHERE lab_id = %s ORDER BY equipment_type, equipment_name"
        result = self.db.fetch_query(query, (lab_id,))
        return pd.DataFrame(result) if result else pd.DataFrame()
    
    # ===== CERTIFICATION OPERATIONS =====
    
    def add_certification(self, lab_id, certification_name, issuing_body, issue_date, expiry_date):
        """Add certification to laboratory"""
        query = """
        INSERT INTO certification (lab_id, certification_name, issuing_body, issue_date, expiry_date)
        VALUES (%s, %s, %s, %s, %s)
        """
        return self.db.execute_query(query, (lab_id, certification_name, issuing_body, issue_date, expiry_date), show_result=True)
    
    def get_certifications_by_lab(self, lab_id):
        """Get all certifications for a laboratory"""
        query = "SELECT * FROM certification WHERE lab_id = %s ORDER BY certification_name"
        result = self.db.fetch_query(query, (lab_id,))
        return pd.DataFrame(result) if result else pd.DataFrame()
    
    # ===== ACCREDITATION OPERATIONS =====
    
    def add_accreditation(self, lab_id, accreditation_name, accrediting_body, standard_code, accreditation_scope, issue_date, expiry_date):
        """Add accreditation to laboratory"""
        query = """
        INSERT INTO accreditation (lab_id, accreditation_name, accrediting_body, standard_code, accreditation_scope, issue_date, expiry_date)
        VALUES (%s, %s, %s, %s, %s, %s, %s)
        """
        return self.db.execute_query(query, (lab_id, accreditation_name, accrediting_body, standard_code, accreditation_scope, issue_date, expiry_date), show_result=True)
    
    def get_accreditations_by_lab(self, lab_id):
        """Get all accreditations for a laboratory"""
        query = "SELECT * FROM accreditation WHERE lab_id = %s ORDER BY accreditation_name"
        result = self.db.fetch_query(query, (lab_id,))
        return pd.DataFrame(result) if result else pd.DataFrame()
    
    def get_all_accredited_labs(self):
        """Get all accredited laboratories"""
        query = """
        SELECT DISTINCT l.lab_id, l.lab_name, l.contact_email, a.accreditation_name, a.standard_code
        FROM laboratory l
        JOIN accreditation a ON l.lab_id = a.lab_id
        WHERE a.accreditation_status = 'active'
        ORDER BY l.lab_name
        """
        result = self.db.fetch_query(query)
        return pd.DataFrame(result) if result else pd.DataFrame()

# Initialize manager
manager = LaboratoryManager(db)
print('✅ Laboratory Manager class created')

✅ Laboratory Manager class created


## CELL 9: Insert Sample Data

In [29]:
# Sample data - Add laboratories
sample_labs = [
    ('PT Lab Indonesia', 'Leading laboratory for chemical testing', 'info@ptlab.id', '+62-21-1234567', 'www.ptlab.id', 2010),
    ('Bio Testing Center Jakarta', 'Biological and microbiological testing', 'contact@biotesting.id', '+62-21-7654321', 'www.biotesting.id', 2015),
    ('Pharma Quality Lab', 'Pharmaceutical quality assurance', 'qc@pharmaqualitylab.id', '+62-274-123456', 'www.pharmaqualitylab.id', 2012),
    ('Food Safety Laboratory', 'Food and beverage analysis', 'info@foodsafety.id', '+62-31-555666', 'www.foodsafety.id', 2008),
    ('Environmental Lab Bandung', 'Environmental and water quality testing', 'env@labkualitas.id', '+62-22-888999', 'www.labkualitas.id', 2018),
]

print('Adding sample laboratories...')
for lab in sample_labs:
    manager.add_laboratory(*lab)
    
print('\n✅ Sample laboratories added')

Adding sample laboratories...
✅ Query executed (1 rows affected)
✅ Query executed (1 rows affected)
✅ Query executed (1 rows affected)
✅ Query executed (1 rows affected)
✅ Query executed (1 rows affected)

✅ Sample laboratories added


In [30]:
# Add sample locations
sample_locations = [
    (1, 'DKI Jakarta', 'Jakarta Pusat', 'Menteng', 'Jl. Gatot Subroto No. 50', '12930', '-6.2194', '106.8207'),
    (2, 'DKI Jakarta', 'Jakarta Timur', 'Cakung', 'Jl. Raya Pulo Alung', '13910', '-6.1754', '106.9567'),
    (3, 'DI Yogyakarta', 'Yogyakarta', 'Mantrijeron', 'Jl. Prof. Soepomo 85', '55143', '-7.7956', '110.4062'),
    (4, 'Jawa Timur', 'Surabaya', 'Tambaksari', 'Jl. Raya Darmo 66', '60188', '-7.2575', '112.7521'),
    (5, 'Jawa Barat', 'Bandung', 'Sukajadi', 'Jl. Ir. H. Djuanda 157', '40135', '-6.8957', '107.6107'),
]

print('Adding locations...')
for location in sample_locations:
    manager.add_location(*location)
    
print('\n✅ Sample locations added')

Adding locations...
✅ Query executed (1 rows affected)
✅ Query executed (1 rows affected)
✅ Query executed (1 rows affected)
✅ Query executed (1 rows affected)
✅ Query executed (1 rows affected)

✅ Sample locations added


In [31]:
# Add sample services
sample_services = [
    ('Heavy Metal Analysis', 'Testing for lead, mercury, cadmium', 'Chemical Testing', 7, 500000),
    ('Microbial Count', 'Bacterial and fungal identification', 'Biological Testing', 5, 350000),
    ('HPLC Analysis', 'High-Performance Liquid Chromatography', 'Chemical Testing', 10, 750000),
    ('Sterility Test', 'Pharmaceutical sterility assurance', 'Pharmaceutical Testing', 14, 600000),
    ('pH & Conductivity', 'Water quality measurement', 'Environmental Testing', 3, 150000),
    ('Pesticide Residue', 'Detection in food samples', 'Food Safety', 8, 550000),
    ('Protein Analysis', 'Protein content quantification', 'Biological Testing', 5, 400000),
]

print('Adding services...')
for service in sample_services:
    manager.add_service(*service)
    
print('\n✅ Sample services added')

Adding services...
✅ Query executed (1 rows affected)
✅ Query executed (1 rows affected)
✅ Query executed (1 rows affected)
✅ Query executed (1 rows affected)
✅ Query executed (1 rows affected)
✅ Query executed (1 rows affected)
✅ Query executed (1 rows affected)

✅ Sample services added


In [32]:
# Link services to labs
sample_links = [
    (1, 1, 550000, 50),     # Lab 1 - Heavy Metal
    (1, 3, 800000, 30),     # Lab 1 - HPLC
    (2, 2, 400000, 40),     # Lab 2 - Microbial
    (2, 7, 420000, 50),     # Lab 2 - Protein
    (3, 4, 650000, 20),     # Lab 3 - Sterility
    (3, 3, 850000, 25),     # Lab 3 - HPLC
    (4, 6, 600000, 35),     # Lab 4 - Pesticide
    (5, 5, 160000, 60),     # Lab 5 - pH & Conductivity
    (5, 1, 520000, 40),     # Lab 5 - Heavy Metal
]

print('Linking services to laboratories...')
for link in sample_links:
    manager.link_service_to_lab(*link)
    
print('\n✅ Services linked to laboratories')

Linking services to laboratories...
✅ Query executed (1 rows affected)
✅ Query executed (1 rows affected)
✅ Query executed (1 rows affected)
✅ Query executed (1 rows affected)
✅ Query executed (1 rows affected)
✅ Query executed (1 rows affected)
✅ Query executed (1 rows affected)
✅ Query executed (1 rows affected)
✅ Query executed (1 rows affected)

✅ Services linked to laboratories


## CELL 10: Verify Data

In [33]:
# Verify all data
print('=== DATABASE VERIFICATION ===')
print('\n📋 All Laboratories:')
labs = manager.get_all_laboratories()
print(labs[['lab_id', 'lab_name', 'contact_email', 'operational_status']].to_string())
print(f'\nTotal: {len(labs)} laboratories')

=== DATABASE VERIFICATION ===

📋 All Laboratories:
   lab_id                    lab_name           contact_email operational_status
0       2  Bio Testing Center Jakarta   contact@biotesting.id        operational
1       5   Environmental Lab Bandung      env@labkualitas.id        operational
2       4      Food Safety Laboratory      info@foodsafety.id        operational
3       3          Pharma Quality Lab  qc@pharmaqualitylab.id        operational
4       1            PT Lab Indonesia           info@ptlab.id        operational

Total: 5 laboratories


In [34]:
print('\n📋 All Services:')
services = manager.get_all_services()
print(services[['service_id', 'service_name', 'service_category', 'estimated_turnaround_days']].to_string())
print(f'\nTotal: {len(services)} services')


📋 All Services:
   service_id          service_name        service_category  estimated_turnaround_days
0           2       Microbial Count      Biological Testing                          5
1           7      Protein Analysis      Biological Testing                          5
2           1  Heavy Metal Analysis        Chemical Testing                          7
3           3         HPLC Analysis        Chemical Testing                         10
4           5     pH & Conductivity   Environmental Testing                          3
5           6     Pesticide Residue             Food Safety                          8
6           4        Sterility Test  Pharmaceutical Testing                         14

Total: 7 services


In [35]:
print('\n📍 All Provinces:')
provinces = manager.get_all_provinces()
for prov in provinces:
    print(f'  - {prov}')
print(f'\nTotal: {len(provinces)} provinces')


📍 All Provinces:
  - DI Yogyakarta
  - DKI Jakarta
  - Jawa Barat
  - Jawa Timur

Total: 4 provinces


In [36]:
print('\n✅ DATABASE SETUP COMPLETE!')
print('\n📊 Summary:')
print(f'  ✓ 9 Tables created')
print(f'  ✓ {len(labs)} Laboratories')
print(f'  ✓ {len(services)} Services')
print(f'  ✓ {len(provinces)} Provinces')
print(f'\nNow run: streamlit run app.py')


✅ DATABASE SETUP COMPLETE!

📊 Summary:
  ✓ 9 Tables created
  ✓ 5 Laboratories
  ✓ 7 Services
  ✓ 4 Provinces

Now run: streamlit run app.py


## CELL 11: Keep Connection Open

In [37]:
# Keep connection open for Streamlit app
print('✅ Database connection ready for Streamlit app')
print('Do NOT disconnect while running Streamlit!')

✅ Database connection ready for Streamlit app
Do NOT disconnect while running Streamlit!
